# Sanity 2: Residual ReLU Block (Analytical vs Numerical)
Compares analytic determinant and condition number against autograd Jacobian extraction.

In [ ]:
import torch
import matplotlib.pyplot as plt

eps = 0.1
W1 = torch.eye(2)
W2 = eps * torch.eye(2)
b1 = torch.zeros(2)

def block(h):
    z = W1 @ h + b1
    return h + W2 @ torch.relu(z)

def jacobian_numeric(h):
    h = h.clone().requires_grad_(True)
    return torch.autograd.functional.jacobian(block, h)

test_points = [torch.tensor([1.0, 1.0]), torch.tensor([-1.0, -1.0]), torch.tensor([1.0, -1.0])]
labels = ['both active', 'both inactive', 'mixed']
numeric_logdets = []
analytic_logdets = []

for h, lbl in zip(test_points, labels):
    J = jacobian_numeric(h)
    sign, logabs = torch.linalg.slogdet(J)
    active = (h > 0).to(torch.float32)
    analytic_diag = 1.0 + eps * active
    analytic_log = torch.log(analytic_diag).sum().item()
    numeric_logdets.append(float(logabs.item()))
    analytic_logdets.append(analytic_log)
    cond = torch.linalg.cond(J).item()
    print(f'{lbl:14s} | numeric log|det|={float(logabs):+.6f} | analytic={analytic_log:+.6f} | cond={cond:.4f}')

x = range(len(labels))
plt.figure(figsize=(6, 4))
plt.plot(x, numeric_logdets, marker='o', label='numeric')
plt.plot(x, analytic_logdets, marker='s', label='analytic')
plt.xticks(list(x), labels, rotation=15)
plt.ylabel('log|det J|')
plt.title('Analytical vs Numerical log|det|')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()